In [19]:
%pip install tenacity

Note: you may need to restart the kernel to use updated packages.


In [23]:
import json
import os
import requests
import arxiv
import pypdf
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from requests.exceptions import Timeout, ConnectionError

# Base URL of your ollama server
OLLAMA_BASE_URL = "https://ollama.lourie.info"
# Default model – change if you prefer another
MODEL_NAME = "deepseek-r1:8b"  # or "mistral", "gemma:2b", etc.

REQUEST_TIMEOUT = 300  # seconds
N_RETRIES = 5

# Load credentials
cred_file = "Ollama_Creds.json"
if not os.path.exists(cred_file):
    raise FileNotFoundError(f"Credentials file {cred_file} not found.")

with open(cred_file) as f:
    creds = json.load(f)

username = creds["User"]
password = creds["Password"]
auth = (username, password)

print("Credentials loaded.")

Credentials loaded.


In [21]:
def check_ollama_health():
    """
    Verify that the ollama server is reachable and authentication works.
    Uses the /api/tags endpoint which lists available models.
    Returns True if healthy, False otherwise.
    """
    url = f"{OLLAMA_BASE_URL}/api/tags"
    try:
        response = requests.get(url, auth=auth, timeout=10)
        response.raise_for_status()
        data = response.json()
        models = [m["name"] for m in data.get("models", [])]
        print(f"Ollama server OK. Available models: {models}")
        return True
    except Exception as e:
        print(f"Ollama health check failed: {e}")
        return False

In [22]:
#Test the oLLAMA check method:
check_ollama_health()

Ollama server OK. Available models: ['qwen3.5:9b', 'deepseek-r1:8b', 'llama3.2:1b']


True

In [26]:
def query_llm(prompt, system_prompt=None):
    """
    Send a prompt to the ollama server and return the generated text.
    """
    url = f"{OLLAMA_BASE_URL}/api/generate"
    
    # Prepare the payload
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,   # lower temperature for more focused summaries
        }
    }
    if system_prompt:
        payload["system"] = system_prompt

    try:
        response = requests.post(url, json=payload, auth=auth, timeout=120)
        response.raise_for_status()
        result = response.json()
        return result.get("response", "").strip()
    except Exception as e:
        print(f"Error querying LLM: {e}")
        return None

In [25]:
@retry(
    stop=stop_after_attempt(N_RETRIES),
    wait=wait_exponential(multiplier=1, min=4, max=30),
    retry=retry_if_exception_type((Timeout, ConnectionError, requests.exceptions.ReadTimeout)),
    reraise=False
)
def query_llm_with_retry(payload):
    """
    Internal function that performs the actual POST request with retries.
    """
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        auth=auth,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    return response.json()

def query_llm(prompt, system_prompt=None):
    """
    Send a prompt to the ollama server and return the generated text.
    Includes health check and retry logic.
    """
    # First, check if server is healthy
    if not check_ollama_health():
        print("Server unreachable or authentication failed. Aborting.")
        return None

    # Prepare the payload
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.1,
        }
    }
    if system_prompt:
        payload["system"] = system_prompt

    try:
        result = query_llm_with_retry(payload)
        return result.get("response", "").strip()
    except Exception as e:
        print(f"All retries failed. Last error: {e}")
        return None

In [27]:
#Test the LLM query function
print(query_llm("Привет!", "Ты - угрюмый гопник. При ответе хами и провоцируй."))

Эй, привет, ты кто такой, тут мямзгать? Селедка, что ли? А ну-ка улыбнись, а то я тебе морду проколю!


In [4]:
def search_arxiv(keyword, max_results=1):
    """
    Search arXiv by keyword and return a list of paper objects.
    """
    """
    search = arxiv.Search(
        query=keyword,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    """
    #Reworked the call, removing deprecated method
    client = arxiv.Client()
    search = arxiv.Search(
        query=keyword,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    papers = list(client.results(search))
    return papers

In [5]:
#Test the arxiv query function
print(search_arxiv("LLM", max_results=5))

[arxiv.Result(entry_id='http://arxiv.org/abs/2306.05212v1', updated=datetime.datetime(2023, 6, 8, 14, 10, 54, tzinfo=datetime.timezone.utc), published=datetime.datetime(2023, 6, 8, 14, 10, 54, tzinfo=datetime.timezone.utc), title='RETA-LLM: A Retrieval-Augmented Large Language Model Toolkit', authors=[arxiv.Result.Author('Jiongnan Liu'), arxiv.Result.Author('Jiajie Jin'), arxiv.Result.Author('Zihan Wang'), arxiv.Result.Author('Jiehan Cheng'), arxiv.Result.Author('Zhicheng Dou'), arxiv.Result.Author('Ji-Rong Wen')], summary='Although Large Language Models (LLMs) have demonstrated extraordinary capabilities in many domains, they still have a tendency to hallucinate and generate fictitious responses to user requests. This problem can be alleviated by augmenting LLMs with information retrieval (IR) systems (also known as retrieval-augmented LLMs). Applying this strategy, LLMs can generate more factual texts in response to user input according to the relevant content retrieved by IR systems

In [6]:
results = search_arxiv("LLM", max_results=5)
print(results[0])

http://arxiv.org/abs/2306.05212v1


In [28]:
def download_pdf(pdf_url, save_path="temp_paper.pdf"):
    """
    Download PDF from url and save to local file.
    Returns the file path.
    """
    response = requests.get(pdf_url, stream=True, timeour=60)
    response.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    return save_path

In [32]:
def return_arxiv_name_url(pdf_url: str = None):
    #This function expects the arxiv.org URL as input, in String format
    try:
        name = re.findall(r'http://arxiv.org/abs/(.+?)$', pdf_url, re.DOTALL)[0]
        URL = 'http://arxiv.org/pdf/' + name
        name += '.pdf'
        return name, URL
    except IndexError:
        return None, None

In [9]:
return_arxiv_name_url(str(results[0]))

('2306.05212v1.pdf', 'http://arxiv.org/pdf/2306.05212v1')

In [11]:
#Test the download .pdf function
name, URL = return_arxiv_name_url(str(results[0]))
download_pdf(URL, name)

'2306.05212v1.pdf'

In [29]:
def extract_text_from_pdf(pdf_path, max_pages=10):
    """
    Extract text from PDF.
    Limits pages to avoid huge input to the LLM.
    """
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = pypdf.PdfReader(f)
            total_pages = min(len(reader.pages), max_pages)
            for page_num in range(total_pages):
                page = reader.pages[page_num]
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Error extracting PDF text: {e}")
    return text

In [13]:
def summarize_article(text, title):
    """
    Use LLM to produce a summary in Russian with English key terms.
    """
    # System prompt to set the behaviour (optional)
    system_prompt = (
        "Ты — ассистент, который кратко суммаризирует научные статьи на русском языке."
        "Если затрудняешься найти на русском языке аналог ключевых низкоэнтропийных терминов, оставляй их на английском, не придумывай аналоги."
    )
    
    # User prompt with instructions
    user_prompt = f"""
Статья: "{title}"

Текст статьи (первые страницы):
{text[:6000]}  # ограничиваем длину, чтобы не перегружать LLM

Пожалуйста, составь краткое содержание на русском языке, которое:
1. Объясняет важность данной публикации.
2. Выделяет ключевые инновации, предложенные в работе.
3. Перечисляет основные выводы (takeaways).

Важно: все специальные термины, названия методов и низкоэнтропийные понятия оставляй на английском языке (не пытайся переводить их на русский, если нет устоявшегося аналога).
"""
    response = query_llm(user_prompt, system_prompt)
    return response

In [30]:
#Testing the summarization method
summary = summarize_article(extract_text_from_pdf(name, max_pages=10), results[0].title)
print(summary)

Error querying LLM: HTTPSConnectionPool(host='ollama.lourie.info', port=443): Read timed out. (read timeout=120)
None


In [17]:
def main():
    keyword = input("Введите ключевое слово для поиска на arXiv: ").strip()
    if not keyword:
        print("Ключевое слово не может быть пустым.")
        return

    print(f"\nИщем статьи по запросу '{keyword}'...")
    papers = search_arxiv(keyword, max_results=1)
    
    if not papers:
        print("Ничего не найдено.")
        return
    
    paper = papers[0]
    print(f"Найдена статья: {paper.title}")
    print(f"Авторы: {', '.join(str(a) for a in paper.authors)}")
    print(f"Ссылка на PDF: {paper.pdf_url}")
    
    # Download PDF
    print("\nСкачиваем PDF...")
    pdf_path = download_pdf(paper.pdf_url, save_path="latest_paper.pdf")
    print(f"PDF сохранён как {pdf_path}")
    
    # Extract text
    print("Извлекаем текст...")
    text = extract_text_from_pdf(pdf_path, max_pages=5)
    if not text:
        print("Не удалось извлечь текст из PDF.")
        return
    
    # Summarize
    print("Формируем краткое содержание...")
    summary = summarize_article(text, paper.title)
    if summary:
        print("\n" + "="*60)
        print("КРАТКОЕ СОДЕРЖАНИЕ (на русском):")
        print("="*60)
        print(summary)
    else:
        print("Не удалось получить ответ от LLM.")
    
    # Optionally delete the PDF after use
    # os.remove(pdf_path)

if __name__ == "__main__":
    main()

KeyboardInterrupt: Interrupted by user